In [ ]:
!pip install transformers torch sentence-transformers outlines

In [ ]:
!pip install -U weaviate-client

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("hf_token")
from huggingface_hub import login
login(token=hf_token)

In [ ]:
import os
import numpy as np
import torch
import weaviate

from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
from weaviate.classes.init import Auth
from weaviate.classes.query import MetadataQuery
from pydantic import BaseModel
from typing import List, Dict

# --- Config ---
MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"
weaviate_url = user_secrets.get_secret("weaviate_url")
weaviate_key = user_secrets.get_secret("weaviate_key")

In [ ]:
# --- Models ---
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

In [ ]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
# --- Weaviate client ---
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=weaviate_url,
    auth_credentials=Auth.api_key(weaviate_key),
)

In [ ]:
def get_embedding(text: str) -> list:
    vector = np.array(embedder.encode(text), dtype="float32")
    return (vector / np.linalg.norm(vector)).tolist()

In [ ]:
def ask_llm(messages: List[Dict[str, str]], max_new_tokens: int = 512) -> str:
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)

    # Decode only the newly generated tokens
    generated = outputs[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

In [ ]:
def rag_search(query: str, top_k: int = 3) -> List[Dict]:
    collection = client.collections.get("ProductFAQ")
    results = collection.query.hybrid(
        query=query,
        vector=get_embedding(query),
        limit=top_k,
        return_properties=["question", "answer", "category"],
        return_metadata=MetadataQuery(score=True),
    ).objects

    print("Results =", results)

    return [
        {
            "question": r.properties["question"],
            "answer": r.properties["answer"],
            "category": r.properties["category"],
            "score": r.metadata.score,
        }
        for r in results
    ]

In [ ]:
def generate_response(query: str) -> str:
    try:
        results = rag_search(query)
    except Exception as e:
        print(f"RAG search failed: {e}")
        results = []

    context = "\n".join(r["answer"] for r in results) if results else "No reference found."

    messages = [
        {
            "role": "system",
            "content": (
                "You are a MyanmarNet assistant. Answer using the references below.\n\n"
                f"References:\n{context}\n\n"
                "Be concise. If unsure, ask a clarifying question."
            ),
        },
        {"role": "user", "content": query},
    ]

    return ask_llm(messages)

In [ ]:
generate_response("What are A Wa Thone packs?")

In [ ]:
generate_response("Do you offer A Myan Thone Packs?")

In [ ]:
generate_response("I want to buy A Wa Thone and A Myan Thone Packs. Where can I buy? ")